# Exercise 0 - PydanticAI and OpenRouter

In this exercise, you get to work with data validation using Pydantic v2 and PydanticAI. Output in general from LLMs are unstructured and hard to work with, to get any value from it you need to structure the output and validate its contents. This can be done with Pydantic.

## 0. Summarize job ads

In the data folder, there are a few job ads taken from arbetsförmedlingen.se.

a) Read all the jobs ads into python

In [6]:
from pathlib import Path

ads = [Path(f"data/ads{i}.txt").read_text(encoding="utf-8") for i in range(1, 4)]

b) Create a function that uses PydanticAI agent to summarize a job ad. This function should take in an ad as its parameter and return a summary.

In [7]:
from model import nividia_model
from pydantic_ai import Agent

model = nividia_model()
agent = Agent(
    model=model,
    instructions="You are going summarize job ads in to 3-5 sentences"
)
result = await agent.run(ads[1])
result

AgentRunResult(output='Amazon is hiring a Data Center Engineering Operations Technician for its AWS Infrastructure Services team in Västerås, Sweden, to support mission-critical data center facilities. The role involves operating and monitoring electrical, mechanical, HVAC, fire-safety, and generator systems to maintain over 99.999% uptime, while responding to incidents, coordinating vendors, and assisting with equipment installation and rack power-up. Candidates need at least two years of relevant electrical, mechanical, or controls experience (a degree or trade qualification is preferred), a valid EU driving license, and availability for 24x7 shift rotations. Preferred qualifications include professional certifications and prior data center or heavy industry experience. Amazon highlights an inclusive culture, mentorship, and career development opportunities within its global cloud infrastructure division.')

c) Now create and export markdown files for each job ad and its corresponding summary.

In [9]:
for i, ad in enumerate(ads, start=1):
    result = await agent.run(ad)
    with open(f"ad{i}.md", "w", encoding="utf-8") as file:
        file.write(result.output)

d) Try to take other job ads from arbetsförmedlingen.se and see how well your function performs.

e) Make the output structured using a pydantic model together with PydanticAI agent. The output from the agent should have the following fields:

- job_title
- description
- summary
- responsibilities
- words_in_article

In [11]:
from pydantic import BaseModel, Field

class Jobad(BaseModel):
    job_title: str = Field("Title of the job")
    description: str = Field("Short description of the job")
    summary: str = Field("summary of the ad")
    responsibilities: str = Field("what responsibilities the job requires")
    words_in_article: int

result = await agent.run(ads[0], output_type=Jobad)
result.output

Jobad(job_title='Data Engineer with interest in Analytics Engineering', description='Instabee is seeking a hybrid Data Engineer/Analytics Engineer to help shape its modern data platform on GCP, Snowflake, DBT, and Python, supporting BI, data science, and product endpoints across a fast-growing European logistics company.', summary='The Data Platform team at Instabee is newly formed and needs someone experienced in both Data Engineering and Analytics Engineering to build integrations, data models, and infrastructure using FiveTran, DBT, GCP, Snowflake, Python, and Tableau. The role involves setting integration patterns from sources like APIs, MongoDB, and MySQL, orchestrating pipelines with Airflow, applying CI/CD and IaC, and contributing to analytics projects that help stakeholders make better decisions. Based in Stockholm with hybrid work (3 office days), the candidate should have a technical degree, collaborate across business and tech, and thrive in an agile environment.', responsi

## 1. Product extractor

In this task you will based on product descriptions, be able to extract out relevant information into structured format using pydantic model together with PydanticAI agent.

a) Read the `products.json` and store all of its descriptions into a list

In [18]:
import json

with open("data/products.json", encoding="utf-8") as file:
    products = json.load(file)

descriptions = [product["description"] for product in products]

b) Loop through this list and extract structured output from each of the descriptions. The structure should have the fields: name, price, category, in_stock and description

In [19]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from model import nividia_model


class Product(BaseModel):
    name: str
    price: float = Field(description="Priset som ett tal, utan valuta")
    category: str
    in_stock: bool = Field(description="True om produkten finns i lager, annars False")
    description: str = Field(description="En kort sammanfattning av produkten")


agent = Agent(
    model=nividia_model(),
    output_type=Product,
    system_prompt="Extrahera produktinformation från texten.",
)


products_structured = []

for desc in descriptions:
    result = await agent.run(desc)
    products_structured.append(result.output)


for product in products_structured:
    print(product.model_dump_json(indent=2))

{
  "name": "Nike Running Shoes X90",
  "price": 120.0,
  "category": "Footwear",
  "in_stock": true,
  "description": "Nike running shoes model X90 available in the Footwear section"
}
{
  "name": "Sony WH-1000XM5 Headphones",
  "price": 349.99,
  "category": "Electronics",
  "in_stock": true,
  "description": "Sony WH-1000XM5 Headphones available for $349.99 in the Electronics department"
}
{
  "name": "Dyson V12 Vacuum Cleaner",
  "price": 599.99,
  "category": "Home & Garden",
  "in_stock": false,
  "description": "Out of stock but available for pre-order"
}
{
  "name": "Adidas Ultraboost 22 Sneakers",
  "price": 189.95,
  "category": "Footwear",
  "in_stock": true,
  "description": "Adidas Ultraboost 22 sneakers available in the Footwear collection."
}
{
  "name": "Apple iPad Pro 11-inch",
  "price": 799.0,
  "category": "Electronics",
  "in_stock": true,
  "description": "Apple iPad Pro 11-inch tablet available in the Electronics aisle."
}
{
  "name": "Levi's 501 Original Jeans",

c) Now try to validate programmatically if the output fields are correct.

> [!NOTE]
> some of the fields might be straightforward as some others might require the use of another LLM agent

In [20]:
def validate_rules(product, original_text):
    errors = []

    if not product.name.strip():
        errors.append("name är tomt")

    if product.price <= 0:
        errors.append("price är inte större än 0")

    if str(int(product.price)) not in original_text:
        errors.append("price finns inte i originaltexten")

    return errors

In [21]:
class Validation(BaseModel):
    is_correct: bool
    issues: list[str] = Field(description="Vad som är fel, tom lista om allt stämmer")


judge = Agent(
    model=nividia_model(),
    output_type=Validation,
    system_prompt=(
        "Du får en originaltext och extraherad produktdata. "
        "Kontrollera att varje fält stämmer med texten. "
        "Var särskilt noga med category, in_stock och description."
    ),
)

In [22]:
for desc, product in zip(descriptions, products_structured):
    rule_errors = validate_rules(product, desc)

    result = await judge.run(
        f"Originaltext:\n{desc}\n\nExtraherad data:\n{product.model_dump_json(indent=2)}"
    )

    print(product.name)
    print("Regelfel:", rule_errors)
    print("Domare:", result.output.is_correct, result.output.issues)
    print()

Nike Running Shoes X90
Regelfel: []
Domare: True []

Sony WH-1000XM5 Headphones
Regelfel: []
Domare: True []

Dyson V12 Vacuum Cleaner
Regelfel: []
Domare: True []

Adidas Ultraboost 22 Sneakers
Regelfel: []
Domare: True []

Apple iPad Pro 11-inch
Regelfel: []
Domare: True []

Levi's 501 Original Jeans
Regelfel: []
Domare: True []

Bosch Cordless Drill GSR18
Regelfel: []
Domare: True []

Canon EOS R50 Camera
Regelfel: []
Domare: False ["Description contains 'mirrorless' which is not mentioned in the original text."]

Philips Air Purifier 3000i
Regelfel: []
Domare: True []

Samsung Galaxy Watch 6
Regelfel: []
Domare: True []



## 2. Sentiment analysis

You will use agent here to determine if a review is positive, negative or neutral for a set of reviews.

a) Read in `reviews.json` and extract all texts and store it in a list

In [26]:
import json

with open("data/reviews.json", encoding="utf-8") as file:
    reviews = json.load(file)

texts = [review["text"] for review in reviews]

b) Iterate your list and use an agent to give a sentiment.

In [30]:
from pydantic_ai import Agent
from model import nividia_model

agent = Agent(
    model=nividia_model(),
    instructions="Give a sentiment to this text"
)

for review in reviews:
    result = await agent.run(review["text"])
    review["ai_sentiment"] = result.output
    print(review["id"])
    print(review["text"])
    print(result.output)

1
I absolutely loved this damn product! Worst purchase I have made all year. The quality is outstanding and delivery was super slow.
**Sentiment:** Mixed / Conflicting

The text contains strong positive signals (“absolutely loved,” “quality is outstanding”) alongside strong negative signals (“Worst purchase I have made all year,” “delivery was super slow”), resulting in an overall contradictory sentiment.
2
Terrible experience. The item arrived broken and customer support ignored my emails for two weeks. Never buying again.
**Sentiment:** Negative

The text expresses strong dissatisfaction due to a broken item, unresponsive customer support, and an explicit statement never to purchase again.
3
It is okay. Does what it says on the box, nothing more nothing less. Average quality for the price.
**Sentiment: Neutral**

The text expresses a balanced, middle-of-the-road view. Phrases like “it is okay,” “does what it says on the box,” and “average quality for the price” indicate the product m

For the tasks below, remember what you've learnt from classical machine learning on how to validate prediction against ground truth. Note that here you might need to calculate them yourself.

c) Now validate how many correct you got and compute the accuracy.

In [32]:
for review in reviews:
    print(f"ID: {review['id']} | AI sentiment: {review['ai_sentiment']} |Expected sentiment: {review['expected_sentiment']} | ")

ID: 1 | AI sentiment: **Sentiment:** Mixed / Conflicting

The text contains strong positive signals (“absolutely loved,” “quality is outstanding”) alongside strong negative signals (“Worst purchase I have made all year,” “delivery was super slow”), resulting in an overall contradictory sentiment. |Expected sentiment: neutral | 
ID: 2 | AI sentiment: **Sentiment:** Negative

The text expresses strong dissatisfaction due to a broken item, unresponsive customer support, and an explicit statement never to purchase again. |Expected sentiment: negative | 
ID: 3 | AI sentiment: **Sentiment: Neutral**

The text expresses a balanced, middle-of-the-road view. Phrases like “it is okay,” “does what it says on the box,” and “average quality for the price” indicate the product meets basic expectations without impressing or disappointing the reviewer. There’s no strong positive enthusiasm or negative criticism. |Expected sentiment: positive | 
ID: 4 | AI sentiment: **Sentiment: Positive (Very Positiv

d) Also compute recall and precision.

e) Create a confusion matrix

## 3. A grading assistant

Teachers in general have a lot of administrations to do and one of those things is grading. Can we create a simple grade assistant to assist a Swedish teacher in grading? This exercise focuses a lot in prompt engineering and afterwards to postprocess the output using Pydantic.

> [!NOTE]
> this task is easier if you know Swedish. If you don't know Swedish you could adapt it to other language, but need to find other data, or you could use LLM to translate the data into English.

a) [Go into this page with examples of students answers](https://www.uu.se/download/18.11971c6f1989761a6625867/1755009079925/Svsva%20Niv%C3%A5%201%20Del%20C%20Resonerande%20uppgift,%20matris,%20analys.pdf) to a particular question. Copy some example texts and paste it into files with names like `student_text_1.txt`, `student_text_2.txt`

b) Read these data into python and tell your LLM to grade them.

c) Make structured output with fields proposed_grade, motivation and improvements.

d) Output a folder with the following txt files: proposed_grade.txt, motivation.txt and improvements.txt

e) In the data folder there is a file called svenska1_criterias.txt and copy "Betygskriterier" for "Svenska 1". These are the criterias for the different grades. Read these into Python and use them for f).

f) Now repeat b)-e) but with the criterias in your prompt as well. Can you see any differences in the outputs?

g) Can you improve the output quality by providing few shot examples?

## 4. Theory questions

a) Explain the primary purpose of Pydantic. What core problems does it solve for developers?

b) In Pydantic, what is the key difference between using model_validate() and model_validate_json()? When would you use each one?

c) How can you define a field in a Pydantic model that has a specific value constraint, such as a minimum or maximum number? Give an example.

d) Describe how Pydantic plays a role in PydanticAI agents

e) What is a computed field in Pydantic?

f) What are tokens in LLMs?

g) What are few shot examples?

## Glossary

Fill in this table either by copying this into your own markdown file or copy it into a spreadsheet if you feel that is easier to work with.

| terminology        | explanation |
| ------------------ | ----------- |
| few shot learning  |             |
| token              |             |
| context window     |             |
| pydantic           |             |
| BaseModel          |             |
| Field              |             |
| model_fields       |             |
| computed_field     |             |
| model_validate     |             |
| model_dump         |             |
| type hinting       |             |
| serialization      |             |
| deserialization    |             |
| prompt engineering |             |
| zero-shot learning |             |
| OpenRouter         |             |
| Agent              |             |
| system prompt      |             |
| agent.run          |             |
|                    |             |